# Test telephone outcome composition without repeated weighting

**Data and method:** the national benchmark, inbound sensitivity and the nested 1,456-practice outcome-complete cohort.

**Purpose:** calculate the authoritative NHS-aligned three-coordinate representation of four CBT outcome shares and compare 17-feature, raw 21-feature and ILR 20-feature models on the same 1,456 practices.

The reference outcome parts are strictly positive. Each row is closed to one and no pseudocount is added. Coordinate names and the orthonormal basis encode dealt-versus-missed, answered-versus-IVR/callback and IVR-versus-callback balances.

In [1]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
CONFIG_PATH = ROOT / 'configs' / 'reference_apr2025_mar2026.json'
from gpap2.config import load_config
REFERENCE_CONFIG = load_config(CONFIG_PATH)
AUTHORITY_MANIFEST = REFERENCE_CONFIG.resolve(REFERENCE_CONFIG.authority_checksum_file)
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 120)
pd.set_option('display.precision', 6)


## Method contract

This stage inherits the 1,456-practice outcome-complete cohort and compares the inbound 17-feature baseline, raw 21-feature outcome representation and NHS-aligned 20-feature ILR representation under identical model controls. Composition, transformation and output tables are generated through the tested [composition implementation](../src/gpap2/composition.py), [analysis implementation](../src/gpap2/analysis.py), [preprocessing implementation](../src/gpap2/preprocessing.py) and [reporting helpers](../src/gpap2/notebook_reporting.py).

In [2]:
import numpy as np
import pandas as pd
from gpap2.analysis import add_nhs_outcome_coordinates, compare_outcome_models
from gpap2.composition import NHS_OUTCOME_ILR_BASIS, inverse_nhs_outcome_ilr, validate_orthonormal_basis
from gpap2.config import load_config
from gpap2.contracts import NHS_OUTCOME_ILR_NAMES, OUTCOME_SHARE_COLUMNS
from gpap2.io import read_contract_csv, validate_authority_file
from gpap2.notebook_reporting import (
    build_comparison_contract_table,
    build_composition_contract_table,
    build_feature_contract_table,
    build_model_contract_table,
)

config = load_config(CONFIG_PATH)
source = config.resolve(config.input_directory) / config.specification('cbt_outcome_raw_21').source_file
outcomes = read_contract_csv(source)
shares = outcomes.loc[:, OUTCOME_SHARE_COLUMNS].to_numpy(dtype=float)
working, closed = add_nhs_outcome_coordinates(outcomes)
coordinates = working.loc[:, NHS_OUTCOME_ILR_NAMES].to_numpy(dtype=float)
validate_orthonormal_basis(NHS_OUTCOME_ILR_BASIS)
reconstruction_error = float(np.max(np.abs(inverse_nhs_outcome_ilr(coordinates) - closed)))
pseudocount_used = False

composition_audit = build_composition_contract_table(
    shares,
    closed,
    NHS_OUTCOME_ILR_BASIS,
    OUTCOME_SHARE_COLUMNS,
    NHS_OUTCOME_ILR_NAMES,
    reconstruction_error,
    pseudocount_used=pseudocount_used,
)
composition_audit

,measure,observed
0,outcome part order,cbt_answered_share_cbt003 | cbt_missed_share |...
1,all source parts strictly positive,True
2,original row-sum minimum,0.921953
3,original row-sum median,0.999645
4,original row-sum maximum,1.004244
5,closed row-sum maximum error from one,0.0
6,pseudocount used,False
7,ILR coordinate order,ilr_dealt_vs_missed | ilr_answered_vs_ivr_call...
8,basis orthonormality maximum error,0.0
9,inverse reconstruction maximum error,0.0


In [3]:
basis_table = pd.DataFrame(
    NHS_OUTCOME_ILR_BASIS,
    index=NHS_OUTCOME_ILR_NAMES,
    columns=OUTCOME_SHARE_COLUMNS,
)
basis_table

,cbt_answered_share_cbt003,cbt_missed_share,cbt_ivr_share,cbt_callback_request_share
ilr_dealt_vs_missed,0.288675,-0.866025,0.288675,0.288675
ilr_answered_vs_ivr_callback,0.816497,0.000000,-0.408248,-0.408248
ilr_ivr_vs_callback,0.000000,0.000000,0.707107,-0.707107


In [4]:
comparison = compare_outcome_models(outcomes, config)
specification_rows = []
for specification in ['cbt_inbound_17', 'cbt_outcome_raw_21', 'cbt_outcome_nhs_ilr_20']:
    feature_table = build_feature_contract_table(config, specification)
    specification_rows.append({
        'specification': specification,
        'features': len(feature_table),
        'log1p features': ' | '.join(feature_table.loc[feature_table['transformation'].eq('log1p'), 'feature']),
        'unchanged features': ' | '.join(feature_table.loc[feature_table['transformation'].eq('unchanged'), 'feature']),
    })
pd.DataFrame(specification_rows)

,specification,features,log1p features,unchanged features
0,cbt_inbound_17,17,ocs_submissions_per_1000_patient_months | gpad...,ocs_clinical_share | ocs_administrative_share ...
1,cbt_outcome_raw_21,21,ocs_submissions_per_1000_patient_months | gpad...,ocs_clinical_share | ocs_administrative_share ...
2,cbt_outcome_nhs_ilr_20,20,ocs_submissions_per_1000_patient_months | gpad...,ocs_clinical_share | ocs_administrative_share ...


In [5]:
build_model_contract_table(
    config,
    comparison.prepared['cbt_outcome_nhs_ilr_20'],
)

,setting,value,runtime_source
0,algorithm,K-Means,src/gpap2/models.py
1,clusters (k),3,reference configuration
2,initialisation,k-means++,src/gpap2/models.py
3,n_init,100,reference configuration
4,max_iter,500,reference configuration
5,random_state,2026,reference configuration
6,implementation,lloyd,reference configuration
7,centering,median,reference configuration
8,scaling,interquartile range (IQR),reference configuration
9,label alignment,maximum-agreement Hungarian assignment,src/gpap2/models.py


In [6]:
authority_path = ROOT / 'outputs' / 'validation' / 'analytical_regression_results.csv'
validate_authority_file(authority_path, AUTHORITY_MANIFEST)
authority = pd.read_csv(authority_path)
observed = comparison.comparisons.merge(authority, on=['reference', 'candidate'], suffixes=('_recomputed', '_authority'))
for field in ['adjusted_rand_index', 'normalised_mutual_information', 'aligned_agreement']:
    assert np.allclose(observed[f'{field}_recomputed'], observed[f'{field}_authority'], atol=1e-12, rtol=0)
assert (observed['reassigned_practices_recomputed'] == observed['reassigned_practices_authority']).all()
observed[['reference', 'candidate', 'adjusted_rand_index_recomputed', 'normalised_mutual_information_recomputed', 'aligned_agreement_recomputed', 'reassigned_practices_recomputed']]

,reference,candidate,adjusted_rand_index_recomputed,normalised_mutual_information_recomputed,aligned_agreement_recomputed,reassigned_practices_recomputed
0,cbt_inbound_17,cbt_outcome_nhs_ilr_20,0.894915,0.838160,0.964286,52
1,cbt_inbound_17,cbt_outcome_raw_21,0.248685,0.287697,0.499313,729
2,cbt_outcome_raw_21,cbt_outcome_nhs_ilr_20,0.242190,0.280143,0.495192,735


In [7]:
nhs_row = comparison.comparisons.loc[comparison.comparisons['candidate'].eq('cbt_outcome_nhs_ilr_20') & comparison.comparisons['reference'].eq('cbt_inbound_17')].iloc[0]
raw_row = comparison.comparisons.loc[comparison.comparisons['candidate'].eq('cbt_outcome_raw_21') & comparison.comparisons['reference'].eq('cbt_inbound_17')].iloc[0]
assert (shares > 0).all() and pseudocount_used is False
assert reconstruction_error < 1e-12
from gpap2.analysis import canonical_assignment_sha256
assignment_hash = canonical_assignment_sha256(
    comparison.assignments,
    config.identifier,
    'cbt_outcome_nhs_ilr_20_aligned_to_inbound_17',
)
expected_nhs = authority.loc[
    authority['reference'].eq(nhs_row['reference']) & authority['candidate'].eq(nhs_row['candidate'])
].iloc[0]
expected_raw = authority.loc[
    authority['reference'].eq(raw_row['reference']) & authority['candidate'].eq(raw_row['candidate'])
].iloc[0]
for observed_row, expected_row in [(nhs_row, expected_nhs), (raw_row, expected_raw)]:
    for metric in ['adjusted_rand_index', 'normalised_mutual_information', 'aligned_agreement']:
        assert np.isclose(observed_row[metric], expected_row[metric], atol=1e-12, rtol=0)
    assert int(observed_row['reassigned_practices']) == int(expected_row['reassigned_practices'])
assert assignment_hash == expected_nhs['canonical_aligned_assignment_sha256']
comparison_contract = build_comparison_contract_table(
    comparison.comparisons,
    comparison.diagnostics,
    {'cbt_outcome_nhs_ilr_20': assignment_hash},
)
comparison_contract

,reference,candidate,adjusted_rand_index,normalised_mutual_information,aligned_agreement,reassigned_practices,reference_silhouette,candidate_silhouette,canonical_aligned_assignment_sha256
0,cbt_inbound_17,cbt_outcome_nhs_ilr_20,0.894915,0.838160,0.964286,52,0.118416,0.101971,8E2B9618DE15BB363EA28F2946F7DDBD530845AC413507...
1,cbt_inbound_17,cbt_outcome_raw_21,0.248685,0.287697,0.499313,729,0.118416,0.099228,
2,cbt_outcome_raw_21,cbt_outcome_nhs_ilr_20,0.242190,0.280143,0.495192,735,0.099228,0.101971,8E2B9618DE15BB363EA28F2946F7DDBD530845AC413507...


## Decision

The NHS-aligned 20-feature ILR model is the preferred outcome-representation sensitivity. The raw 21-feature model remains a representation comparator. Neither restricted-cohort model replaces the national model.

**What this establishes:** The outcome-representation result contributes to the robustness evidence alongside inbound, algorithmic, feature and temporal comparisons.